In [1]:
# ==============================================================================
# CELL 0: INSTALL PIPELINE DEPENDENCIES
# ==============================================================================
# 1. Object Detection & Tracking
!pip install -q ultralytics

# 2. Vector Database (Hardware Accelerated)
!pip install -q faiss-gpu

# 3. Facial Recognition Meta-Framework
!pip install -q deepface tf-keras

# 4. Body Re-Identification (OSNet) - Must be built from source
!pip install -q git+https://github.com/KaiyangZhou/deep-person-reid.git

print("✅ All forensic dependencies installed successfully!")

^C


ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


✅ All forensic dependencies installed successfully!


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> [6 lines of output]
      Traceback (most recent call last):
        File "<string>", line 2, in <module>
        File "<pip-setuptools-caller>", line 35, in <module>
        File "C:\Users\prince\AppData\Local\Temp\pip-req-build-aff0o5xi\setup.py", line 5, in <module>
          from Cython.Build import cythonize
      ModuleNotFoundError: No module named 'Cython'
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [ ]:
!pip install -q faiss-cpu ultralytics deepface tf-keras
!pip install -q git+https://github.com/KaiyangZhou/deep-person-reid.git
print("installation completed")

In [ ]:
# ==============================================================================
# CELL 1: THE FULL 18-STEP FORENSIC INGESTION (FINAL SECURE BUILD)
# ==============================================================================
import os, cv2, torch, faiss, shutil, sys, pickle, logging, contextlib
import numpy as np
from collections import defaultdict
from tqdm.notebook import tqdm
from ultralytics import YOLO
import torchreid
from sklearn.cluster import KMeans
from deepface import DeepFace

# 🤫 SILENCE LOGS
os.environ['ORT_LOGGING_LEVEL'] = '3'         
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'      
logging.getLogger('insightface').setLevel(logging.ERROR)

# ⚙️ CONFIGURATION
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
import glob
try:
    INPUT_PATH = glob.glob('/kaggle/input/forensic-input/*.mp4')[0]
except:
    INPUT_PATH = "/kaggle/input/datasets/directorprince/video-lalpari/Laal Pari - Housefull 5_HD-(HDvideo9).mp4"
BASE_OUTPUT_DIR = "/kaggle/working/forensic_master"
try:
    QUERY_IMAGE = [f for f in glob.glob('/kaggle/input/forensic-input/*') if f.endswith('.jpg') or f.endswith('.jpeg') or f.endswith('.png')][0]
except:
    QUERY_IMAGE = "/kaggle/input/datasets/directorprince/query-image4/Screenshot 2026-02-23 170233.png"
MIN_WIDTH, MIN_HEIGHT = 50, 100    
BLUR_THRESHOLD = 80.0              

if os.path.exists(BASE_OUTPUT_DIR): shutil.rmtree(BASE_OUTPUT_DIR)
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

if os.path.isfile(INPUT_PATH):
    VIDEO_FOLDER = os.path.dirname(INPUT_PATH)
    video_files = [os.path.basename(INPUT_PATH)]
else:
    VIDEO_FOLDER = INPUT_PATH
    video_files = sorted([f for f in os.listdir(VIDEO_FOLDER) if f.endswith('.mp4')])

# 🤖 LOAD MODELS
print(f"📦 Loading 18-Stage AI Pipeline on {DEVICE.upper()}...")
yolo = YOLO("yolov8m-seg.pt") 
osnet_model = torchreid.models.build_model(name="osnet_x1_0", num_classes=1000, pretrained=True).to(DEVICE).eval()
DeepFace.build_model("ArcFace") 

# 🎯 AUTO-QUERY EXTRACTOR
if not os.path.exists(QUERY_IMAGE):
    print("⚠️ Custom query not found. Auto-extracting...")
    first_video_path = os.path.join(VIDEO_FOLDER, video_files[0])
    cap = cv2.VideoCapture(first_video_path)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        res = yolo.track(frame, persist=True, verbose=False)
        if res[0].boxes is not None and res[0].boxes.id is not None:
            for box in res[0].boxes.xyxy.cpu().numpy():
                x1, y1, x2, y2 = map(int, box)
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
                if (x2-x1) > 80 and (y2-y1) > 160: 
                    cv2.imwrite(QUERY_IMAGE, frame[y1:y2, x1:x2])
                    break
        if os.path.exists(QUERY_IMAGE): break
    cap.release()

# 🧩 GLOBAL EXTRACTOR (512 Dimensions)
def get_global_emb(img_bgr):
    s_res = cv2.resize(img_bgr, (128, 256))
    s_rgb = cv2.cvtColor(s_res, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    s_norm = (s_rgb - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    tensor = torch.tensor(np.transpose(s_norm, (2, 0, 1)), dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): 
        feat = osnet_model(tensor).cpu().numpy().flatten()
    return feat / (np.linalg.norm(feat) + 1e-12)

# 🧩 DEEPFACE FACIAL EXTRACTION (512 Dimensions)
def get_face_emb(img_bgr):
    try:
        results = DeepFace.represent(img_path=img_bgr, model_name="ArcFace", detector_backend="retinaface", enforce_detection=False)
        if not results: return None, 0, 0.0
            
        best_face, max_area = None, 0
        for res in results:
            conf = res.get('face_confidence', 0)
            if conf == 0: continue
            fa = res['facial_area']
            area = fa['w'] * fa['h']
            if area > max_area:
                max_area = area
                best_face = res
                
        if best_face is None: return None, 0, 0.0
        emb = np.array(best_face['embedding'], dtype=np.float32)
        return emb, max_area, best_face['face_confidence']
    except Exception:
        return None, 0, 0.0

# 🧩 DYNAMIC FUSION (Total: 1024 Dimensions)
def fuse_embs(global_emb, face_emb, f_area, b_area, f_conf):
    if face_emb is None or (f_area/max(1, b_area)) < 0.05 or f_conf < 0.85:
        fused = np.concatenate([np.zeros(512, dtype=np.float32), global_emb]) 
        return fused / (np.linalg.norm(fused) + 1e-12)
    alpha = 0.7 if (f_area / max(1, b_area)) >= 0.30 else 0.4
    fused = np.concatenate([alpha * face_emb, (1.0 - alpha) * global_emb])
    return fused / (np.linalg.norm(fused) + 1e-12)

# 🏗️ SCANNER LOOP
print("\n🏗️ ENGINE 1: Indexing cameras...")
global_metadata, gallery_protos, proto_to_gid = defaultdict(list), [], []

for v_name in tqdm(video_files, desc="🎥 Total Progress"):
    v_path = os.path.join(VIDEO_FOLDER, v_name)
    cap = cv2.VideoCapture(v_path)
    total_f, fps = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), cap.get(cv2.CAP_PROP_FPS) or 30.0
    f_idx, track_dict = 0, defaultdict(list)
    
    pbar = tqdm(total=total_f, desc=f"   ↳ {v_name}", leave=False)
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        
        if f_idx % max(1, int(fps)) == 0:
            res = yolo.track(frame, persist=True, tracker="botsort.yaml", verbose=False)
            if res[0].boxes is not None and res[0].boxes.id is not None:
                boxes, ids, masks = res[0].boxes.xyxy.cpu().numpy(), res[0].boxes.id.cpu().numpy(), res[0].masks
                
                for i, box in enumerate(boxes):
                    x1, y1, x2, y2 = map(int, box)
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
                    
                    if (x2-x1) < MIN_WIDTH or (y2-y1) < MIN_HEIGHT: continue
                    crop = frame[y1:y2, x1:x2].copy()
                    if crop.size == 0: continue
                        
                    if cv2.Laplacian(cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var() < BLUR_THRESHOLD: 
                        continue
                        
                    global_emb = get_global_emb(crop)
                    f_emb, f_area, f_conf = get_face_emb(crop)
                    final_dna = fuse_embs(global_emb, f_emb, f_area, crop.shape[0]*crop.shape[1], f_conf)
                    
                    tid, g_id = int(ids[i]), f"{v_name}_ID_{int(ids[i])}"
                    track_dict[tid].append(final_dna)
                    
                    global_metadata[g_id].append({'v_path': v_path, 'fps': fps, 'f': f_idx, 'b': (x1,y1,x2,y2), 'e': final_dna})
        f_idx += 1
        pbar.update(1)
    pbar.close(); cap.release()

    for tid, embs in track_dict.items():
        arr = np.vstack(embs)
        if len(arr) > 2:
            distances = np.linalg.norm(arr - np.mean(arr, axis=0), axis=1)
            arr = arr[distances <= (np.mean(distances) + 2*np.std(distances))]
            
        if len(arr) == 0: continue
        
        km = KMeans(n_clusters=max(1, min(3, len(arr))), random_state=42, n_init=10).fit(arr)
        for center in km.cluster_centers_:
            gallery_protos.append(center / (np.linalg.norm(center) + 1e-12))
            proto_to_gid.append(f"{v_name}_ID_{tid}")

if not gallery_protos: sys.exit("❌ CRITICAL: No people found.")

with open('/kaggle/working/global_metadata.pkl', 'wb') as f: pickle.dump(global_metadata, f)
with open('/kaggle/working/gallery_protos.pkl', 'wb') as f: pickle.dump(gallery_protos, f)
with open('/kaggle/working/proto_to_gid.pkl', 'wb') as f: pickle.dump(proto_to_gid, f)
print("✅ 18-Step Database Locked!")

In [ ]:
# ==============================================================================
# CELL 2: ENGINE 2 & 3 - SEARCH & RENDER (FINAL SECURE BUILD)
# ==============================================================================
import pickle, faiss, cv2, os, sys, torch, contextlib
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from deepface import DeepFace

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUERY_IMAGE = "/kaggle/input/datasets/directorprince/query-image4/Screenshot 2026-02-23 170233.png"
BASE_OUTPUT_DIR = "/kaggle/working/forensic_master"

# 🤖 RE-INITIALIZE MODELS
import torchreid
osnet_model = torchreid.models.build_model(name="osnet_x1_0", num_classes=1000, pretrained=True).to(DEVICE).eval()
DeepFace.build_model("ArcFace")

def get_global_emb(img_bgr):
    s_res = cv2.resize(img_bgr, (128, 256))
    s_rgb = cv2.cvtColor(s_res, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    s_norm = (s_rgb - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
    tensor = torch.tensor(np.transpose(s_norm, (2, 0, 1)), dtype=torch.float32).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): 
        feat = osnet_model(tensor).cpu().numpy().flatten()
    return feat / (np.linalg.norm(feat) + 1e-12)

def get_face_emb(img_bgr):
    try:
        results = DeepFace.represent(img_path=img_bgr, model_name="ArcFace", detector_backend="retinaface", enforce_detection=False)
        if not results: return None, 0, 0.0
            
        best_face, max_area = None, 0
        for res in results:
            conf = res.get('face_confidence', 0)
            if conf == 0: continue
            fa = res['facial_area']
            area = fa['w'] * fa['h']
            if area > max_area:
                max_area = area
                best_face = res
                
        if best_face is None: return None, 0, 0.0
        emb = np.array(best_face['embedding'], dtype=np.float32)
        return emb, max_area, best_face['face_confidence']
    except Exception:
        return None, 0, 0.0

def fuse_embs(global_emb, face_emb, f_area, b_area, f_conf):
    if face_emb is None or (f_area/max(1, b_area)) < 0.05 or f_conf < 0.85:
        fused = np.concatenate([np.zeros(512, dtype=np.float32), global_emb]) 
        return fused / (np.linalg.norm(fused) + 1e-12)
    alpha = 0.7 if (f_area / max(1, b_area)) >= 0.30 else 0.4
    fused = np.concatenate([alpha * face_emb, (1.0 - alpha) * global_emb])
    return fused / (np.linalg.norm(fused) + 1e-12)

# --- 1. LOAD DATA ---
try:
    with open('/kaggle/working/global_metadata.pkl', 'rb') as f: global_metadata = pickle.load(f)
    with open('/kaggle/working/gallery_protos.pkl', 'rb') as f: gallery_protos = pickle.load(f)
    with open('/kaggle/working/proto_to_gid.pkl', 'rb') as f: proto_to_gid = pickle.load(f)
except FileNotFoundError:
    sys.exit("❌ Error: Run Cell 1 first.")

# --- 2. ENGINE 2: FORENSIC SEARCH (1024 Dimensions) ---
index = faiss.IndexFlatIP(1024)  
gallery_matrix = np.vstack(gallery_protos).astype("float32")
index.add(gallery_matrix)

q_img = cv2.imread(QUERY_IMAGE)
if q_img is None or q_img.size == 0: sys.exit("❌ Query image corrupted.")

q_global = get_global_emb(q_img)
q_f_emb, q_f_area, q_f_conf = get_face_emb(q_img)
q_emb = fuse_embs(q_global, q_f_emb, q_f_area, q_img.shape[0]*q_img.shape[1], q_f_conf).reshape(1, -1).astype("float32")

total_gallery = gallery_matrix.shape[0]
k_search = min(100, total_gallery)

sims, idxs = index.search(q_emb, k=k_search)
verified_hits = {}

for i, p_idx in enumerate(idxs[0]):
    if p_idx == -1: continue
    g_id = proto_to_gid[p_idx]
    c_sim = sims[0][i]
    verified_hits[g_id] = max(verified_hits.get(g_id, 0), c_sim)

# --- GRID DISPLAY ---
if not verified_hits: sys.exit("⚠️ No verified matches found.")

display_ids = sorted(verified_hits, key=verified_hits.get, reverse=True)[:19]
cols = 5
rows = max(2, (len(display_ids) + 1 + cols - 1) // cols)
fig, axes = plt.subplots(rows, cols, figsize=(20, 4 * rows), squeeze=False)
axes_flat = axes.flatten()

axes_flat[0].imshow(cv2.cvtColor(q_img, cv2.COLOR_BGR2RGB))
axes_flat[0].set_title("🎯 QUERY", color='blue', fontweight='bold'); axes_flat[0].axis('off')

for i, g_id in enumerate(display_ids):
    idx = i + 1
    best = max(global_metadata[g_id], key=lambda x: np.dot(q_emb, x['e'])[0])
    cap = cv2.VideoCapture(best['v_path']); cap.set(cv2.CAP_PROP_POS_FRAMES, best['f']); ret, f = cap.read()
    if ret:
        x1, y1, x2, y2 = best['b']
        x1, y1 = max(0, x1), max(0, y1)
        crop = f[y1:y2, x1:x2]
        if crop.size > 0: 
            axes_flat[idx].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            axes_flat[idx].set_title(f"{g_id}\nScore: {verified_hits[g_id]:.2f}", color='green', fontsize=9)
    axes_flat[idx].axis('off'); cap.release()
for j in range(len(display_ids) + 1, len(axes_flat)): axes_flat[j].axis('off')
plt.tight_layout(); plt.show()

# --- 3. ENGINE 3: EXTRACTION & ACCURACY METRICS ---
target_input = input(f"🎯 Enter Global IDs (comma separated): ").strip()
target_ids = [x.strip() for x in target_input.split(",") if x.strip() in global_metadata]

if not target_ids: sys.exit("⚠️ No valid IDs selected.")

total_shown = len(display_ids)
correct_selections = len(target_ids)
hit_rate = (correct_selections / total_shown) * 100
print(f"\n📊 SYSTEM ACCURACY REPORT:")
print(f" -> Top-{total_shown} Retrieval Precision: {hit_rate:.1f}% ({correct_selections}/{total_shown} correct matches)")

for g_id in tqdm(target_ids, desc="🎬 Rendering Evidence"):
    # 🔒 TEMPORAL LOCK: Sorts metadata by frame index to prevent interpolation crashes
    meta = sorted(global_metadata[g_id], key=lambda x: x['f'])
    v_path, fps = meta[0]['v_path'], meta[0]['fps']
    
    f_list = [m['f'] for m in meta]
    f_start = max(0, min(f_list) - int(fps))
    f_end = max(f_list) + int(fps)
    
    frame_map = {m['f']: m for m in meta}
    for i in range(len(meta)-1):
        gap = meta[i+1]['f'] - meta[i]['f']
        if 1 < gap < (int(fps) * 5): 
            for j in range(1, gap):
                frac = j/gap
                b1, b2 = meta[i]['b'], meta[i+1]['b']
                interp_b = (int(b1[0]+(b2[0]-b1[0])*frac), int(b1[1]+(b2[1]-b1[1])*frac),
                            int(b1[2]+(b2[2]-b1[2])*frac), int(b1[3]+(b2[3]-b1[3])*frac))
                frame_map[meta[i]['f']+j] = {'b': interp_b, 'e': meta[i]['e'], 'smoothed': True}
    
    cap = cv2.VideoCapture(v_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, f_start)
    fourcc = cv2.VideoWriter_fourcc(*'XVID') 
    out_path = os.path.join(BASE_OUTPUT_DIR, f"evidence_{g_id.replace('.','_')}.avi")
    out = cv2.VideoWriter(out_path, fourcc, int(fps), (int(cap.get(3)), int(cap.get(4))))
    
    frame_scores = [] 
    for f_num in range(f_start, f_end):
        ret, f = cap.read()
        if not ret: break
        if f_num in frame_map:
            d = frame_map[f_num]
            x1, y1, x2, y2 = d['b']
            color = (0, 165, 255) if d.get('smoothed') else (0, 0, 255)
            cv2.rectangle(f, (x1, y1), (x2, y2), color, 3)
            
            raw_score = np.dot(q_emb, d['e'])[0]
            accuracy_pct = max(0, min(100, int(raw_score * 100)))
            frame_scores.append(accuracy_pct)
            cv2.putText(f, f"ACCURACY: {accuracy_pct}%", (x1, y1-10), 0, 0.6, (0, 255, 0), 2)
        out.write(f)
        
    cap.release()
    out.release()
    
    avg_clip_acc = sum(frame_scores) / len(frame_scores) if frame_scores else 0
    print(f"✅ Saved: {out_path} | Avg Accuracy: {avg_clip_acc:.1f}%")